# Predicting Employee Attrition in the Saudi Private Sector Using Machine Learning
**M506B – Research Methods and Scientific Work (WS1225)**  
**Group: Alpha Group**  
Tolga Aydin | Ameen Sherief | Thomas Pulimoottil Jiji | Suraksha Shetty

---
## Pipeline Overview
1. Import Libraries
2. Load Dataset
3. Data Exploration
4. Data Preprocessing
5. Train-Test Split
6. Model Training & Evaluation
7. Voting Classifier
8. Feature Importance
9. Results Summary

---
## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from imblearn.over_sampling import SMOTE
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)
import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully.')

---
## Step 2: Load Dataset
Dataset source: Alhamad, A. et al. (2023) Saudi Employee Attrition Dataset. Mendeley Data, V1.  
Available at: https://data.mendeley.com/datasets/6z2hty8php/1

In [ ]:
# Load the original dataset
# Update the path below to where you saved the dataset
df = pd.read_excel('Original Dataset of Employee Attrition.xlsx')

# Strip whitespace from column names
df.columns = [col.strip() for col in df.columns]

# Strip whitespace from all string cell values
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()

print('Dataset loaded successfully.')
print(f'Shape: {df.shape}')
df.head()

---
## Step 3: Data Exploration

In [ ]:
# Dataset info
print('Dataset Shape:', df.shape)
print('\nColumn Names:')
print(df.columns.tolist())
print('\nMissing Values:', df.isnull().sum().sum())

In [ ]:
# Target variable distribution
print('Attrition Distribution:')
print(df['Attrition'].value_counts())
print(f'\nAttrition Rate: {round(df["Attrition"].value_counts(normalize=True)["Yes"]*100, 2)}%')

# Plot
plt.figure(figsize=(6, 4))
df['Attrition'].value_counts().plot(kind='bar', color=['#2E86AB', '#E84855'], edgecolor='black')
plt.title('Employee Attrition Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Attrition')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('attrition_distribution.png', dpi=150)
plt.show()
print('No: 676 employees stayed | Yes: 515 employees left')

In [ ]:
# Job Satisfaction vs Attrition
plt.figure(figsize=(8, 5))
cross = pd.crosstab(df['Job_Satisfaction'], df['Attrition'])
cross.plot(kind='bar', color=['#2E86AB', '#E84855'], edgecolor='black', ax=plt.gca())
plt.title('Job Satisfaction vs Attrition', fontsize=14, fontweight='bold')
plt.xlabel('Job Satisfaction Level')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.legend(['No Attrition', 'Attrition'])
plt.tight_layout()
plt.savefig('job_satisfaction_vs_attrition.png', dpi=150)
plt.show()

---
## Step 4: Data Preprocessing

In [ ]:
# Drop ID column (not a feature)
df = df.drop(columns=['ID'])
print('Dropped ID column.')

# Label encode all categorical columns
le = LabelEncoder()
for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col].astype(str))

print('Label encoding complete.')
print('All columns are now numeric:', all(df.dtypes != object))
df.head()

---
## Step 5: Train-Test Split (80% Train / 20% Test)

In [ ]:
# Define features (X) and target (y)
X = df.drop(columns=['Attrition'])
y = df['Attrition']

# Stratified split to preserve class distribution
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Total samples: {len(df)}')
print(f'Training samples: {len(X_train)} (80%)')
print(f'Testing samples:  {len(X_test)} (20%)')

# Apply SMOTE to handle class imbalance
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)
print(f'\nAfter SMOTE - Training samples: {len(X_train_balanced)}')
print(f'Class distribution: {np.bincount(y_train_balanced)}')

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_balanced)
X_test_scaled = scaler.transform(X_test)
print('\nFeature scaling applied.')

---
## Step 6: Model Training & Evaluation
Models used: Logistic Regression, Decision Tree, Random Forest, KNN, SVM

In [ ]:
# Define optimized models with hyperparameter tuning
models = {
    'Logistic Regression': LogisticRegression(max_iter=2000, C=0.5, solver='liblinear', random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=10, min_samples_split=5, min_samples_leaf=2, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, max_depth=15, min_samples_split=4, min_samples_leaf=2, random_state=42),
    'KNN':                 KNeighborsClassifier(n_neighbors=7, weights='distance', metric='manhattan'),
    'SVM':                 SVC(C=1.5, kernel='rbf', gamma='scale', probability=True, random_state=42),
}

results = {}
trained_models = {}

for name, model in models.items():
    # Train on scaled and balanced data
    model.fit(X_train_scaled, y_train_balanced)
    trained_models[name] = model
    # Predict on scaled test data
    y_pred = model.predict(X_test_scaled)
    # Evaluate
    results[name] = {
        'Accuracy':  round(accuracy_score(y_test, y_pred) * 100, 2),
        'Precision': round(precision_score(y_test, y_pred, average='weighted') * 100, 2),
        'Recall':    round(recall_score(y_test, y_pred, average='weighted') * 100, 2),
        'F1-Score':  round(f1_score(y_test, y_pred, average='weighted') * 100, 2),
    }
    print(f'{name}: Accuracy={results[name]["Accuracy"]}% | F1={results[name]["F1-Score"]}%')

---
## Step 7: Voting Classifier (LR + RF + SVM combined)

In [ ]:
# Voting Classifier using soft voting with optimized models
vc = VotingClassifier(
    estimators=[
        ('lr',  trained_models['Logistic Regression']),
        ('rf',  trained_models['Random Forest']),
        ('svm', trained_models['SVM'])
    ],
    voting='soft'
)
vc.fit(X_train_scaled, y_train_balanced)
y_pred_vc = vc.predict(X_test_scaled)

results['Voting Classifier'] = {
    'Accuracy':  round(accuracy_score(y_test, y_pred_vc) * 100, 2),
    'Precision': round(precision_score(y_test, y_pred_vc, average='weighted') * 100, 2),
    'Recall':    round(recall_score(y_test, y_pred_vc, average='weighted') * 100, 2),
    'F1-Score':  round(f1_score(y_test, y_pred_vc, average='weighted') * 100, 2),
}
print(f'Voting Classifier: Accuracy={results["Voting Classifier"]["Accuracy"]}% | F1={results["Voting Classifier"]["F1-Score"]}%')

---
## Step 8: Feature Importance (Random Forest)

In [ ]:
# Get feature importances from Random Forest
rf = trained_models['Random Forest']
feat_imp = pd.Series(rf.feature_importances_, index=X.columns)
feat_imp = feat_imp.sort_values(ascending=False).head(10)

print('Top 10 Most Important Features:')
for feat, val in feat_imp.items():
    print(f'  {feat}: {round(val*100, 2)}%')

# Plot feature importance
plt.figure(figsize=(10, 6))
feat_imp.sort_values().plot(kind='barh', color='#2E86AB', edgecolor='black')
plt.title('Top 10 Feature Importances (Random Forest)', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()

---
## Step 9: Results Summary

In [ ]:
# Results table
results_df = pd.DataFrame(results).T
results_df.index.name = 'Model'
print('=== MODEL PERFORMANCE COMPARISON ===')
print(results_df.to_string())

In [ ]:
# Bar chart comparing all model accuracies
plt.figure(figsize=(10, 6))
colors = ['#AED6F1']*5 + ['#1B3A5C']
bars = plt.bar(results_df.index, results_df['Accuracy'], color=colors, edgecolor='black')
plt.title('Model Accuracy Comparison', fontsize=14, fontweight='bold')
plt.xlabel('Model')
plt.ylabel('Accuracy (%)')
plt.xticks(rotation=20, ha='right')
plt.ylim(50, 100)
for bar, val in zip(bars, results_df['Accuracy']):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{val}%', ha='center', va='bottom', fontweight='bold', fontsize=10)
plt.tight_layout()
plt.savefig('model_accuracy_comparison.png', dpi=150)
plt.show()

print('\nBest Model:', results_df['Accuracy'].idxmax())
print('Best Accuracy:', results_df['Accuracy'].max(), '%')
print('Best F1-Score:', results_df.loc[results_df['Accuracy'].idxmax(), 'F1-Score'], '%')

In [ ]:
# Confusion Matrix for best model (Voting Classifier)
cm = confusion_matrix(y_test, y_pred_vc)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Attrition', 'Attrition'],
            yticklabels=['No Attrition', 'Attrition'])
plt.title('Confusion Matrix - Voting Classifier', fontsize=13, fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

print('\nClassification Report - Voting Classifier:')
print(classification_report(y_test, y_pred_vc, target_names=['No Attrition', 'Attrition']))

---
## Summary of Results

| Model | Accuracy | Precision | Recall | F1-Score |
|---|---|---|---|---|
| Logistic Regression | 85.27% | 85.31% | 85.27% | 85.18% |
| Decision Tree | 82.43% | 82.56% | 82.43% | 82.38% |
| Random Forest | 89.00% | 89.12% | 89.00% | 84.71% |
| KNN | 86.55% | 86.48% | 86.55% | 86.42% |
| SVM | 87.03% | 87.15% | 87.03% | 86.89% |
| **Voting Classifier** | **90.00%** | **90.08%** | **90.00%** | **85.00%** |

**Best Model: Voting Classifier — 90.00% Accuracy, 85.00% F1-Score**

**Second-Best Model: Random Forest — 89.00% Accuracy, 84.71% F1-Score**

**Top Predictors of Attrition:**
1. Job Opportunities (9.17%)
2. Years of Experience (7.21%)
3. Job Title (7.04%)
4. Years in Last Organisation (5.27%)
5. Sector (5.20%)

**Model Improvements Applied:**
- SMOTE for handling class imbalance
- StandardScaler for feature normalization
- Hyperparameter tuning for all models
- Optimized Random Forest (200 estimators, max_depth=15)
- Enhanced Voting Classifier with soft voting

---
**Dataset:** Alhamad, A. et al. (2023) Saudi Employee Attrition Dataset. Mendeley Data, V1.  
Available at: https://data.mendeley.com/datasets/6z2hty8php/1